In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json

# Carregando os dados do arquivo JSON
with open('events_dataset.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)

# Preparação dos dados
# O sckit-learn espera que os dados estejam em formato de texto, então vamos combinar as colunas relevantes em uma única string para cada evento
df['tags_string'] = df['tags'].apply(lambda x: ' '.join(x))

# Treinamento do modelo TF-IDF: transformando palavras em matrizes
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['tags_string'])

# Similaridade de cosseno: calculando a similaridade entre os eventos
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Função para recomendar eventos com base em um evento de entrada
def recomendar_eventos(titulo_evento, df, cosine_sim):
    # Encontrar o ID do evento de entrada
    idx = df.index[df['title'] == titulo_evento][0]
    
    # Obter as similaridades de cosseno para o evento de entrada
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Ordenar os eventos com base na similaridade
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Obtem o top 3 (excluindo o próprio evento)
    top_3 = sim_scores[1:4]

    # Retornar os títulos dos eventos recomendados
    event_indices = [i[0] for i in top_3]
    return df.iloc[event_indices][['title', 'tags']]

# Exemplo de uso
evento_alvo = 'Google I/O 2026'
print(f"Recomendações para o evento: {evento_alvo}")
display(recomendar_eventos(evento_alvo, df, cosine_sim))


Recomendações para o evento: Google I/O 2026


,title,tags
6,Data Science e IA Summit,"[data science, machine learning, pandas, llm, ai]"
0,Web Summit Rio 2026,"[inovação, startups, tecnologia, networking, n..."
2,Apple WWDC 2026,"[ios, swift, machine learning, mobile, design]"


In [4]:
import pickle

# Salvando o modelo TF-IDF e a matriz de similaridade
cerebro_ia = {
    'vectorizer': tfidf,
    'matriz_similaridade': cosine_sim
}

# Salvando o modelo em um arquivo pickle
with open('modelo_recomendacao.pkl', 'wb') as arquivo:
    pickle.dump(cerebro_ia, arquivo)
